# Part 1: 양자화와 다양한 데이터 타입 - 실습

- 실습 1: 비트와 데이터 표현의 기초
- 실습 2: 부동소수점 정밀도 이해하기
- 실습 3: 모델 메모리 크기 계산하기

In [1]:
import torch
import numpy as np
print(f"PyTorch 버전: {torch.__version__}")

PyTorch 버전: 2.9.0+cpu


---
## 실습 1: 비트와 데이터 표현의 기초

### 학습 목표
- 다양한 데이터 타입의 비트 수와 표현 범위 이해
- Signed vs Unsigned 정수의 차이 이해
- 오버플로우 현상 직접 확인

### 1-1. 데이터 타입별 메모리 크기 확인

In [6]:
# 부동소수점 타입들
float_types = [
    ("FP64 (double)", torch.float64),
    ("FP32 (float)", torch.float32),
    ("FP16 (half)", torch.float16),
    ("BF16 (bfloat16)", torch.bfloat16),
]

print("=" * 60)
print("부동소수점 타입 비교")
print("=" * 60)

# 예) 부동소수점(float) 타입이 바뀌면 같은 π(파이) 값도 다르게 저장된다
for name, dtype in float_types:
    tensor = torch.tensor([3.14159265358979], dtype=dtype)
    print(f"{name:20} | 크기: {tensor.element_size()}바이트 | 비트: {tensor.element_size() * 8}bit | 값: {tensor.item():.10f}")

부동소수점 타입 비교
FP64 (double)        | 크기: 8바이트 | 비트: 64bit | 값: 3.1415926536
FP32 (float)         | 크기: 4바이트 | 비트: 32bit | 값: 3.1415927410
FP16 (half)          | 크기: 2바이트 | 비트: 16bit | 값: 3.1406250000
BF16 (bfloat16)      | 크기: 2바이트 | 비트: 16bit | 값: 3.1406250000


In [8]:
# 정수 타입들
int_types = [
    ("INT64 (long)", torch.int64),
    ("INT32 (int)", torch.int32),
    ("INT16 (short)", torch.int16),
    ("INT8 (char)", torch.int8),
    ("UINT8 (unsigned)", torch.uint8),
]

print("=" * 60)
print("정수 타입 비교")
print("=" * 60)

# 예) 정수 42의 메모리 사용
for name, dtype in int_types:
    tensor = torch.tensor([42], dtype=dtype)
    print(f"{name:20} | 메모리 크기: {tensor.element_size()}바이트 | 비트: {tensor.element_size() * 8}bit")

정수 타입 비교
INT64 (long)         | 메모리 크기: 8바이트 | 비트: 64bit
INT32 (int)          | 메모리 크기: 4바이트 | 비트: 32bit
INT16 (short)        | 메모리 크기: 2바이트 | 비트: 16bit
INT8 (char)          | 메모리 크기: 1바이트 | 비트: 8bit
UINT8 (unsigned)     | 메모리 크기: 1바이트 | 비트: 8bit


### 1-2. Signed vs Unsigned 정수 표현 범위

In [9]:
# INT8 (Signed): -128 ~ 127
print("INT8 (Signed 8-bit):")
print(f"  최소값: {torch.iinfo(torch.int8).min}")
print(f"  최대값: {torch.iinfo(torch.int8).max}")
print(f"  표현 가능한 값의 개수: {2**8} (= 256개)")

print()

# UINT8 (Unsigned): 0 ~ 255
print("UINT8 (Unsigned 8-bit):")
print(f"  최소값: {torch.iinfo(torch.uint8).min}")
print(f"  최대값: {torch.iinfo(torch.uint8).max}")
print(f"  표현 가능한 값의 개수: {2**8} (= 256개)")

INT8 (Signed 8-bit):
  최소값: -128
  최대값: 127
  표현 가능한 값의 개수: 256 (= 256개)

UINT8 (Unsigned 8-bit):
  최소값: 0
  최대값: 255
  표현 가능한 값의 개수: 256 (= 256개)


### 🎯 실습 1-1: FP16 정밀도 부족을 직접 확인하기 (1씩 증가가 안 되는 순간)
- 어느 순간부터 +1 해도 값이 안 바뀌는 현상이 발생함
- FP16은 비트 수가 작아서(16bit) 표현할 수 있는 실수 값이 제한적이다. 표현 가능한 숫자가 드문드문 존재함. 그래서 숫자가 연속적으로 저장되지 않고, 띄엄띄엄(계단처럼) 저장된다.
- 값이 커질수록 표현 가능한 숫자 사이 간격(step)이 커진다. 그 결과 어떤 구간에서는 +1 정도 변화가 너무 작아 반올림되어 값이 안 바뀐다.

In [12]:
import torch

x = torch.tensor([2048.0], dtype=torch.float16)
print("초기값:", x.item())

# 2048에서 1씩 늘려 보면서 실제 값이 바뀌는지 확인
for i in range(1, 20):
    y = x + i
    print(f"2048 + {i:2} = {y.item()}")

초기값: 2048.0
2048 +  1 = 2048.0
2048 +  2 = 2050.0
2048 +  3 = 2052.0
2048 +  4 = 2052.0
2048 +  5 = 2052.0
2048 +  6 = 2054.0
2048 +  7 = 2056.0
2048 +  8 = 2056.0
2048 +  9 = 2056.0
2048 + 10 = 2058.0
2048 + 11 = 2060.0
2048 + 12 = 2060.0
2048 + 13 = 2060.0
2048 + 14 = 2062.0
2048 + 15 = 2064.0
2048 + 16 = 2064.0
2048 + 17 = 2064.0
2048 + 18 = 2066.0
2048 + 19 = 2068.0


---
## 실습 2: 부동소수점 정밀도 이해하기

### 학습 목표
- FP32, FP16, BF16의 정밀도 차이 이해
- 부동소수점의 정밀도 한계 체험
- 양자화 전후 정밀도 손실 계산

### 2-1. 부동소수점 타입별 상세 정보

In [13]:
print("=" * 70)
print("FP32 (Single Precision) - 32비트")
print("=" * 70)
fp32_info = torch.finfo(torch.float32)
print(f"  구조: 1 sign + 8 exponent + 23 mantissa bits")
print(f"  표현 범위: [{fp32_info.min:.2e}, {fp32_info.max:.2e}]")
print(f"  정밀도 (eps): {fp32_info.eps}")
print(f"  유효 자릿수: 약 7자리")

print()
print("=" * 70)
print("FP16 (Half Precision) - 16비트")
print("=" * 70)
fp16_info = torch.finfo(torch.float16)
print(f"  구조: 1 sign + 5 exponent + 10 mantissa bits")
print(f"  표현 범위: [{fp16_info.min:.2e}, {fp16_info.max:.2e}]")
print(f"  정밀도 (eps): {fp16_info.eps}")
print(f"  유효 자릿수: 약 3자리")

print()
print("=" * 70)
print("BF16 (Brain Float 16) - 16비트")
print("=" * 70)
bf16_info = torch.finfo(torch.bfloat16)
print(f"  구조: 1 sign + 8 exponent + 7 mantissa bits")
print(f"  표현 범위: [{bf16_info.min:.2e}, {bf16_info.max:.2e}]")
print(f"  정밀도 (eps): {bf16_info.eps}")
print(f"  유효 자릿수: 약 2자리")
print(f"  특징: FP32와 같은 지수 범위, 딥러닝 학습에 최적화!")

FP32 (Single Precision) - 32비트
  구조: 1 sign + 8 exponent + 23 mantissa bits
  표현 범위: [-3.40e+38, 3.40e+38]
  정밀도 (eps): 1.1920928955078125e-07
  유효 자릿수: 약 7자리

FP16 (Half Precision) - 16비트
  구조: 1 sign + 5 exponent + 10 mantissa bits
  표현 범위: [-6.55e+04, 6.55e+04]
  정밀도 (eps): 0.0009765625
  유효 자릿수: 약 3자리

BF16 (Brain Float 16) - 16비트
  구조: 1 sign + 8 exponent + 7 mantissa bits
  표현 범위: [-3.39e+38, 3.39e+38]
  정밀도 (eps): 0.0078125
  유효 자릿수: 약 2자리
  특징: FP32와 같은 지수 범위, 딥러닝 학습에 최적화!


### 2-2. 정밀도 손실 실험

In [14]:
# 파이 값을 다양한 정밀도로 표현
pi = 3.14159265358979323846

print("π (파이) 값의 정밀도 비교")
print("=" * 50)
print(f"실제 값:  {pi:.20f}")
print()

pi_fp32 = torch.tensor([pi], dtype=torch.float32)
pi_fp16 = torch.tensor([pi], dtype=torch.float16)
pi_bf16 = torch.tensor([pi], dtype=torch.bfloat16)

print(f"FP32:     {pi_fp32.item():.20f}")
print(f"FP16:     {pi_fp16.item():.20f}")
print(f"BF16:     {pi_bf16.item():.20f}")

# 오차 계산
print("\n오차 (|실제값 - 저장값|):")
print(f"FP32 오차: {abs(pi - pi_fp32.item()):.2e}")
print(f"FP16 오차: {abs(pi - pi_fp16.item()):.2e}")
print(f"BF16 오차: {abs(pi - pi_bf16.item()):.2e}")

π (파이) 값의 정밀도 비교
실제 값:  3.14159265358979311600

FP32:     3.14159274101257324219
FP16:     3.14062500000000000000
BF16:     3.14062500000000000000

오차 (|실제값 - 저장값|):
FP32 오차: 8.74e-08
FP16 오차: 9.68e-04
BF16 오차: 9.68e-04


### 2-3. 부동소수점의 함정 - 0.1 + 0.2 != 0.3

In [15]:
# 유명한 부동소수점 문제
a = torch.tensor([0.1], dtype=torch.float32)
b = torch.tensor([0.2], dtype=torch.float32)
c = torch.tensor([0.3], dtype=torch.float32)

result = a + b

print("0.1 + 0.2 == 0.3 ?")
print(f"0.1 = {a.item():.20f}")
print(f"0.2 = {b.item():.20f}")
print(f"0.1 + 0.2 = {result.item():.20f}")
print(f"0.3 = {c.item():.20f}")
print(f"\n0.1 + 0.2 == 0.3: {(result == c).item()}")
print(f"차이: {abs(result.item() - c.item()):.2e}")

0.1 + 0.2 == 0.3 ?
0.1 = 0.10000000149011611938
0.2 = 0.20000000298023223877
0.1 + 0.2 = 0.30000001192092895508
0.3 = 0.30000001192092895508

0.1 + 0.2 == 0.3: True
차이: 0.00e+00


### 🎯 실습 2-1: 다양한 값에서 정밀도 손실 분석

아래 값들을 FP32, FP16, INT8로 변환하고 오차를 계산해보기

In [16]:

test_values = [0.001, 0.5, 1.234, 10.567, 100.999]

print(f"{'원본값':^12} | {'FP32':^12} | {'FP16':^12} | {'INT8':^6} | {'FP16 오차':^10}")
print("-" * 70)

for val in test_values:
    fp32_val = torch.tensor([val], dtype=torch.float32)
    fp16_val = torch.tensor([val], dtype=torch.float16)
    int8_val = torch.tensor([val], dtype=torch.int8)  # 소수점 버림

    fp16_error = abs(val - fp16_val.item())

    print(f"{val:^12.4f} | {fp32_val.item():^12.6f} | {fp16_val.item():^12.6f} | {int8_val.item():^6d} | {fp16_error:^10.2e}")

    원본값      |     FP32     |     FP16     |  INT8  |  FP16 오차  
----------------------------------------------------------------------
   0.0010    |   0.001000   |   0.001000   |   0    |  4.04e-07 
   0.5000    |   0.500000   |   0.500000   |   0    |  0.00e+00 
   1.2340    |   1.234000   |   1.234375   |   1    |  3.75e-04 
  10.5670    |  10.567000   |  10.570312   |   10   |  3.31e-03 
  100.9990   |  100.999001  |  101.000000  |  100   |  1.00e-03 


---
## 실습 3: 모델 메모리 크기 계산하기

### 학습 목표
- 신경망 모델의 파라미터 수 계산
- 데이터 타입에 따른 모델 크기 예측
- 양자화로 인한 메모리 절감 효과 확인

### 3-1. 간단한 MLP 모델 생성

In [17]:
import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleMLP()
print(model)

SimpleMLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)


### 3-2. 파라미터 수 계산

In [18]:
def count_parameters(model):
    """모델의 총 파라미터 수 계산"""
    total_params = 0

    print("레이어별 파라미터 수:")
    print("=" * 50)

    for name, param in model.named_parameters():
        num_params = param.numel()
        total_params += num_params
        print(f"{name:20} | shape: {str(param.shape):20} | params: {num_params:,}")

    print("=" * 50)
    print(f"총 파라미터 수: {total_params:,}")
    return total_params

total_params = count_parameters(model)

레이어별 파라미터 수:
fc1.weight           | shape: torch.Size([256, 784]) | params: 200,704
fc1.bias             | shape: torch.Size([256])    | params: 256
fc2.weight           | shape: torch.Size([10, 256]) | params: 2,560
fc2.bias             | shape: torch.Size([10])     | params: 10
총 파라미터 수: 203,530


### 3-3. 데이터 타입별 모델 크기 계산

In [19]:
def calculate_model_size(num_params, dtype):
    """파라미터 수와 데이터 타입으로 모델 크기 계산"""
    bytes_per_param = {
        'FP64': 8,
        'FP32': 4,
        'FP16': 2,
        'BF16': 2,
        'INT8': 1,
        'INT4': 0.5,
    }

    size_bytes = num_params * bytes_per_param[dtype]
    size_kb = size_bytes / 1024
    size_mb = size_kb / 1024

    return size_bytes, size_kb, size_mb

print(f"\n모델 크기 비교 (총 파라미터: {total_params:,}개)")
print("=" * 60)

dtypes = ['FP32', 'FP16', 'BF16', 'INT8', 'INT4']
fp32_size = None

for dtype in dtypes:
    size_bytes, size_kb, size_mb = calculate_model_size(total_params, dtype)

    if dtype == 'FP32':
        fp32_size = size_bytes
        ratio = 1.0
    else:
        ratio = size_bytes / fp32_size

    print(f"{dtype:6} | {size_bytes:>12,} bytes | {size_kb:>8.2f} KB | {ratio:.0%} of FP32")


모델 크기 비교 (총 파라미터: 203,530개)
FP32   |      814,120 bytes |   795.04 KB | 100% of FP32
FP16   |      407,060 bytes |   397.52 KB | 50% of FP32
BF16   |      407,060 bytes |   397.52 KB | 50% of FP32
INT8   |      203,530 bytes |   198.76 KB | 25% of FP32
INT4   |    101,765.0 bytes |    99.38 KB | 12% of FP32


### 🎯 실습 3-1: GPT-2 모델 크기 계산

GPT-2 모델의 파라미터 수(1.5B = 15억개)를 기준으로 각 데이터 타입별 모델 크기를 계산해보기

In [ ]:
gpt2_params = 1_500_000_000  # 15억 개

print(f"GPT-2 모델 크기 비교 (파라미터: {gpt2_params:,}개)")
print("=" * 70)

dtypes = ['FP32', 'FP16', 'INT8', 'INT4']

for dtype in dtypes:
    size_bytes, size_kb, size_mb = calculate_model_size(gpt2_params, dtype)
    size_gb = size_mb / 1024

    print(f"{dtype:6} | {size_gb:>8.2f} GB")

print("\n💡 인사이트: FP32에서 INT4로 양자화하면 모델 크기가 1/8로 줄어듭니다!")

GPT-2 모델 크기 비교 (파라미터: 1,500,000,000개)
FP32   |     5.59 GB
FP16   |     2.79 GB
INT8   |     1.40 GB
INT4   |     0.70 GB

💡 인사이트: FP32에서 INT4로 양자화하면 모델 크기가 1/8로 줄어듭니다!


### 3-4. 실제 PyTorch 모델 메모리 측정

In [20]:
def get_model_size(model):
    """실제 모델의 메모리 크기 측정"""
    param_size = 0
    buffer_size = 0

    for param in model.parameters():
        param_size += param.nelement() * param.element_size()

    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    total_size = param_size + buffer_size
    return total_size

# FP32 모델
model_fp32 = SimpleMLP()
size_fp32 = get_model_size(model_fp32)

# FP16 모델
model_fp16 = SimpleMLP().half()  # .half()는 FP16으로 변환
size_fp16 = get_model_size(model_fp16)

# BF16 모델
model_bf16 = SimpleMLP().to(torch.bfloat16)
size_bf16 = get_model_size(model_bf16)

print("실제 PyTorch 모델 메모리 크기")
print("=" * 50)
print(f"FP32 모델: {size_fp32:,} bytes ({size_fp32/1024:.2f} KB)")
print(f"FP16 모델: {size_fp16:,} bytes ({size_fp16/1024:.2f} KB)")
print(f"BF16 모델: {size_bf16:,} bytes ({size_bf16/1024:.2f} KB)")
print(f"\n메모리 절감률: {(1 - size_fp16/size_fp32) * 100:.1f}%")

실제 PyTorch 모델 메모리 크기
FP32 모델: 814,120 bytes (795.04 KB)
FP16 모델: 407,060 bytes (397.52 KB)
BF16 모델: 407,060 bytes (397.52 KB)

메모리 절감률: 50.0%


---
### 메모리 절감 효과
| 변환 | 메모리 절감 |
|------|------------|
| FP32 → FP16 | 50% 절감 |
| FP32 → INT8 | 75% 절감 |
| FP32 → INT4 | 87.5% 절감 |
